<a href="https://colab.research.google.com/github/Alice34mung3ai/plp-da-a2-rainfall-advisory/blob/main/rainfall_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
import csv
import numpy as np

counties = ["Nairobi", "Nakuru", "Kisumu", "Mombasa", "Machakos"]

months = np.array(["January", "February", "March", "April", "May", "June",
                   "July", "August", "September", "October", "November", "December"])

clean_rows = [

[64, 56, 92, 219, 176, 35, 18, 24, 31, 61, 165, 104],# Nairobi
[35, 42, 70, 145, 120, 78, 96, 105, 72, 55, 60, 38],# Nakuru

[58, 90, 150, 240, 180, 95, 70, 85, 90, 88, 130, 95],# Kisumu
[30, 15, 55, 150, 240, 110, 90, 65, 60, 90, 100, 70],# Mombasa

]

machakos_raw = ["48", "35", "88", "N/A", "140", " 20 ",
                "9", "12", "", "45", "182", "96"]

In [13]:
def to_int_or_none(value):
    """Return the value as an int, or None if it cannot be converted."""
    try:
        return int(value.strip())
    except ValueError:
        return None


def clean_machakos(raw_values):
    """Convert raw text to ints; impute invalid months with the rounded mean of valid ones."""
    parsed = [to_int_or_none(v) for v in raw_values]
    valid = [v for v in parsed if v is not None]
    fill_value = round(sum(valid) / len(valid))
    imputed_idx = [i for i, v in enumerate(parsed) if v is None]
    cleaned = [fill_value if v is None else v for v in parsed]
    return cleaned, imputed_idx, fill_value


machakos_clean, imputed_idx, fill_value = clean_machakos(machakos_raw)

print("Cleaned Machakos row:", machakos_clean)
print(f"Imputation value (rounded mean of valid readings): {fill_value} mm")
print("Imputed months:", [str(months[i]) for i in imputed_idx])

Cleaned Machakos row: [48, 35, 88, 68, 140, 20, 9, 12, 68, 45, 182, 96]
Imputation value (rounded mean of valid readings): 68 mm
Imputed months: ['April', 'September']


## 3. Combined 5 × 12 array

In [14]:
rain = np.array(clean_rows + [machakos_clean])

print("Shape:", rain.shape)
print("dtype:", rain.dtype)
print(rain)

Shape: (5, 12)
dtype: int64
[[ 64  56  92 219 176  35  18  24  31  61 165 104]
 [ 35  42  70 145 120  78  96 105  72  55  60  38]
 [ 58  90 150 240 180  95  70  85  90  88 130  95]
 [ 30  15  55 150 240 110  90  65  60  90 100  70]
 [ 48  35  88  68 140  20   9  12  68  45 182  96]]


## 4. Annual totals and monthly averages

In [15]:
annual_totals = rain.sum(axis=1)          # one total per county
monthly_avg = rain.mean(axis=0)           # one average per month across counties

print("Annual total rainfall (mm):")
for county, total in zip(counties, annual_totals):
    print(f"  {county:<9} {total}")

print("\nAverage rainfall per month across all counties (mm):")
for month, avg in zip(months, monthly_avg):
    print(f"  {month:<10} {avg:.1f}")

Annual total rainfall (mm):
  Nairobi   1045
  Nakuru    916
  Kisumu    1371
  Mombasa   1075
  Machakos  811

Average rainfall per month across all counties (mm):
  January    47.0
  February   47.6
  March      91.0
  April      164.4
  May        171.2
  June       67.6
  July       56.6
  August     58.2
  September  64.2
  October    67.8
  November   127.4
  December   80.6


## 5. Wettest month per county


In [16]:
wettest_idx = np.argmax(rain, axis=1)
wettest_month = months[wettest_idx]

for county, month, mm in zip(counties, wettest_month, rain.max(axis=1)):
    print(f"{county:<9} wettest month: {month} ({mm} mm)")

Nairobi   wettest month: April (219 mm)
Nakuru    wettest month: April (145 mm)
Kisumu    wettest month: April (240 mm)
Mombasa   wettest month: May (240 mm)
Machakos  wettest month: November (182 mm)


## 6. Dry months (below 50 mm)

In [17]:
dry_mask = rain < 50                      # boolean mask
dry_counts = dry_mask.sum(axis=1)         # True counts as 1

for county, n in zip(counties, dry_counts):
    print(f"{county:<9} dry months: {n}")

Nairobi   dry months: 4
Nakuru    dry months: 3
Kisumu    dry months: 0
Mombasa   dry months: 2
Machakos  dry months: 6


## 7. Planting windows

Adding the array to a copy of itself shifted by one column (`R[:, :-1] + R[:, 1:]`) gives the combined rainfall of every pair of consecutive months in one step. Column *j* of the result is month *j* plus month *j+1*. `argmax` picks the best pair per county.

In [18]:
pair_sums = rain[:, :-1] + rain[:, 1:]    # shape (5, 11)
start_idx = np.argmax(pair_sums, axis=1)
best_totals = pair_sums.max(axis=1)

planting_window = months[start_idx] + " - " + months[start_idx + 1]

print("pair_sums shape:", pair_sums.shape)
for county, window, total in zip(counties, planting_window, best_totals):
    print(f"{county:<9} planting window: {window} ({total} mm combined)")

pair_sums shape: (5, 11)
Nairobi   planting window: April - May (395 mm combined)
Nakuru    planting window: April - May (265 mm combined)
Kisumu    planting window: April - May (420 mm combined)
Mombasa   planting window: April - May (390 mm combined)
Machakos  planting window: November - December (278 mm combined)


## 8. Write `rainfall_summary.csv`

In [19]:
header = ["county", "annual_total_mm", "wettest_month", "dry_months", "planting_window"]

with open("rainfall_summary.csv", "w", newline="") as f:
    writer = csv.writer(f, lineterminator="\n")
    writer.writerow(header)
    for row in zip(counties, annual_totals.tolist(), wettest_month.tolist(),
                   dry_counts.tolist(), planting_window.tolist()):
        writer.writerow(row)

# Read it back to confirm
with open("rainfall_summary.csv", newline="") as f:
    rows = list(csv.reader(f))
print(f"{len(rows) - 1} data rows written")
for r in rows:
    print(r)

5 data rows written
['county', 'annual_total_mm', 'wettest_month', 'dry_months', 'planting_window']
['Nairobi', '1045', 'April', '4', 'April - May']
['Nakuru', '916', 'April', '3', 'April - May']
['Kisumu', '1371', 'April', '0', 'April - May']
['Mombasa', '1075', 'May', '2', 'April - May']
['Machakos', '811', 'November', '6', 'November - December']


# Rainfall Analysis for Five Kenyan Counties

**Scenario:** A farmer-support NGO sends SMS advice on planting windows. This notebook cleans a hand-typed Machakos station record, analyses monthly rainfall for five counties with NumPy, writes `rainfall_summary.csv` and ends with a short planting advisory.

**Sections:** 1. Setup and data · 2. Cleaning Machakos · 3. Combined array · 4. Totals and averages · 5. Wettest month · 6. Dry months · 7. Planting windows · 8. CSV output · 9. Planting Advisory · 10. Data Quality Notes

## Data Quality Notes

- **Source of errors:** The Machakos row was typed by hand and arrived as text. Two entries were invalid: `"N/A"` (April) and an empty string (September).
- **Cleaning method:** Valid entries were converted with `int()`, catching only `ValueError`. The value `" 20 "` (June) had stray spaces; these were stripped and the reading was kept as 20 mm.
- **Imputation:** Both invalid months were replaced with 68 mm, the rounded mean of the 10 valid Machakos readings (67.5 mm). This is a simple estimate and does not reflect seasonality. April falls in the long rains for the other counties, so 68 mm may understate it.
- **Effect on results:** The imputed values add 136 mm to the Machakos annual total (811 mm). Machakos's wettest month (November) and planting window (November - December) do not depend on the imputed months, but its annual total does. The dry-month count of 6 treats both imputed months as not dry (68 mm); if either was truly below 50 mm, the real count would be higher.
- **Not checked:** The four other counties were assumed to be correct. Only one year of data is provided, so these windows describe that year, not a long-term climate average.